In [4]:
# =============================================
# TFM — Optimización de descartes en póker (Montecarlo + ML)
# TODO en una celda — Colab
# Incluye:
#   - Autodetección de rutas
#   - RESULT case-insensitive (RESULT/Result/result)
#   - Compatibilidad OneHotEncoder (sparse_output / sparse)
#   - Reglas: NO romper parejas/tríos/full
#   - Modelo de victoria (RandomForestClassifier) entrenado con tus CSV
#   - Dataset Montecarlo + Modelo de política (RandomForestRegressor)
#   - Uso opcional de best_model*.pkl para clasificar jugadas
#   - Juego por rondas:
#       * entrada SIN comas ("AH 3D 5S 3C QH")
#       * muestra "Las cartas a descartar son: ..."
#       * robo MANUAL (usuario introduce las cartas robadas)
#   - Desempate pro-kicker alto (si P(ganar) ~ iguales) y preferir menos descartes
# =============================================

import os, random, itertools, joblib
from collections import Counter
import numpy as np, pandas as pd

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# --------- Autodetección de rutas ----------
SEARCH_DIRS = [".", "/content", "/mnt/data", "/content/drive/MyDrive"]

def find_file(filename):
    for d in SEARCH_DIRS:
        p = os.path.join(d, filename)
        if os.path.exists(p):
            return os.path.abspath(p)
    return None

def first_writable_dir(candidates):
    for d in candidates:
        try:
            os.makedirs(d, exist_ok=True)
            test = os.path.join(d, "__write_test.tmp")
            with open(test, "w") as f: f.write("ok")
            os.remove(test)
            return d
        except Exception:
            continue
    return os.getcwd()

TRAIN_PATH = find_file("train_transformed-2.csv")
TEST_PATH  = find_file("test_transformed-2.csv")

# best_model opcional: probamos varios nombres comunes
BEST_MODEL_PATH = None
for cand in ["best_model.pkl", "best_model-2.pkl", "best_model_final.pkl"]:
    p = find_file(cand)
    if p:
        BEST_MODEL_PATH = p
        break

if TRAIN_PATH is None or TEST_PATH is None:
    raise FileNotFoundError(
        "No encuentro 'train_transformed-2.csv' y/o 'test_transformed-2.csv'. "
        "Súbelos al panel de archivos de Colab (/content)."
    )

OUT_DIR = first_writable_dir([os.path.dirname(TRAIN_PATH), "/mnt/data", "/content"])
WIN_MODEL_PATH     = os.path.join(OUT_DIR, "win_model.pkl")
POLICY_MODEL_PATH  = os.path.join(OUT_DIR, "montecarlo_policy.pkl")
DATASET_OUT_PATH   = os.path.join(OUT_DIR, "montecarlo_policy_dataset.parquet")

print(f"[IO] train: {TRAIN_PATH}")
print(f"[IO] test : {TEST_PATH}")
print(f"[IO] best : {BEST_MODEL_PATH or '(no encontrado, opcional)'}")
print(f"[IO] out  : {OUT_DIR}")

# ===============================
# Utilidades de cartas y mano
# ===============================
RANK_ORDER = "23456789TJQKA"
RANK_TO_INT = {r:i for i,r in enumerate(RANK_ORDER, start=2)}
INT_TO_RANK = {v:k for k,v in RANK_TO_INT.items()}
SUITS = ["H","D","C","S"]

def parse_card(txt):
    # Entrada oficial: tokens separados por espacio; toleramos comas pegadas a la carta
    t = txt.strip().upper().replace(",", "")
    if len(t) < 2: raise ValueError(f"Carta inválida: '{txt}'")
    rank, suit = t[0], t[1]
    # Permite "10H" => "TH"
    if rank == "1" and len(t)>=3 and t[1]=="0":
        rank, suit = "T", t[2]
    if rank not in RANK_ORDER or suit not in SUITS:
        raise ValueError("Usa ranks 2-9,T,J,Q,K,A y palos H,D,C,S (ej.: AH KD 7C 7D 2S).")
    return (RANK_TO_INT[rank], suit)

def format_card(c): return f"{INT_TO_RANK[c[0]]}{c[1]}"

def deck_without(cards):
    used=set(cards); d=[]
    for r in RANK_TO_INT.values():
        for s in SUITS:
            c=(r,s)
            if c not in used: d.append(c)
    return d

def hand_ranks_suits(hand):
    ranks = sorted([r for r,_ in hand], reverse=True)
    suits = [s for _,s in hand]
    return ranks, suits

def is_straight(ranks_sorted_desc):
    r = sorted(set(ranks_sorted_desc), reverse=True)
    if len(r)<5: return False, None
    for i in range(len(r)-4):
        w=r[i:i+5]
        if w[0]-w[4]==4: return True, w[0]
    # Rueda A-5
    if set([14,5,4,3,2]).issubset(set(r)): return True, 5
    return False, None

def hand_rank(hand):
    """
    Devuelve (categoria, desempate_tuple) donde categoria mayor es mejor:
    9: Escalera de color | 8: Póker | 7: Full | 6: Color | 5: Escalera
    4: Trío | 3: Doble pareja | 2: Pareja | 1: Carta alta
    """
    ranks, suits = hand_ranks_suits(hand)
    cnt = Counter(ranks)
    byc = sorted(cnt.items(), key=lambda x:(x[1],x[0]), reverse=True)
    is_flush = len(set(suits))==1
    is_str, top = is_straight(ranks)
    if is_flush and is_str: return (9,(top,))
    if byc[0][1]==4:
        four=byc[0][0]; kicker=max([r for r in ranks if r!=four]); return (8,(four,kicker))
    if byc[0][1]==3 and byc[1][1]==2:
        return (7,(byc[0][0],byc[1][0]))
    if is_flush: return (6, tuple(sorted(ranks, reverse=True)))
    if is_str:   return (5,(top,))
    if byc[0][1]==3:
        triple=byc[0][0]; kick=sorted([r for r in ranks if r!=triple], reverse=True)
        return (4,(triple,*kick))
    if byc[0][1]==2 and byc[1][1]==2:
        hp=max([x[0] for x in byc if x[1]==2]); lp=min([x[0] for x in byc if x[1]==2])
        kicker=max([r for r in ranks if r not in (hp,lp)])
        return (3,(hp,lp,kicker))
    if byc[0][1]==2:
        pair=byc[0][0]; kick=sorted([r for r in ranks if r!=pair], reverse=True)
        return (2,(pair,*kick))
    return (1, tuple(sorted(ranks, reverse=True)))

def categoria_nombre(cat):
    return {
        9: "Escalera de color",
        8: "Póker",
        7: "Full",
        6: "Color",
        5: "Escalera",
        4: "Trío",
        3: "Doble pareja",
        2: "Pareja",
        1: "Carta alta"
    }.get(cat, f"Categoría {cat}")

def compare_hands(h1,h2):
    return (hand_rank(h1)>hand_rank(h2))-(hand_rank(h1)<hand_rank(h2))

# — Reglas de descarte: NO romper parejas/tríos/full
def discard_masks_allowed(hand):
    ranks,_ = hand_ranks_suits(hand)
    cnt = Counter(ranks)
    involved = {r for r,c in cnt.items() if c>=2}
    allowed = [i for i,(r,_) in enumerate(hand) if r not in involved]
    masks = {tuple([False]*5)}  # permitir pasar (no descartar)
    for k in [1,2,3]:
        for comb in itertools.combinations(allowed,k):
            m=[False]*5
            for i in comb: m[i]=True
            masks.add(tuple(m))
    return sorted(list(masks))

def apply_discard_and_draw(hand, mask, rng, return_drawn=False):
    deck = deck_without(hand); rng.shuffle(deck)
    it=iter(deck); new=[]; drawn=[]
    for i,c in enumerate(hand):
        if mask[i]:
            nc = next(it)
            new.append(nc)
            drawn.append((i, nc))
        else:
            new.append(c)
    return (new, drawn) if return_drawn else new

def apply_discard_and_manual_draw(hand, mask, new_cards):
    """
    Aplica la máscara de descarte y coloca EXACTAMENTE las cartas dadas por el usuario
    en los huecos, en el mismo orden introducido.
    Valida duplicados y que no existan ya en la mano.
    """
    if sum(mask) != len(new_cards):
        raise ValueError(f"Debes introducir exactamente {sum(mask)} cartas nuevas.")
    seen = set(hand)
    converted = []
    for c in new_cards:
        cc = parse_card(c)
        if cc in seen or cc in converted:
            raise ValueError(f"Carta duplicada o ya en mano: {c}")
        converted.append(cc)
    new_hand = []
    it = iter(converted)
    for i,c in enumerate(hand):
        new_hand.append(next(it) if mask[i] else c)
    return new_hand, [(i, new_hand[i]) for i,b in enumerate(mask) if b]

def simulate_win_probability(hand, mask, sims=2000, rng=None):
    rng = rng or random.Random()
    w=t=l=0
    for _ in range(sims):
        nh = apply_discard_and_draw(hand, mask, rng)
        deck = deck_without(nh); rng.shuffle(deck); opp = deck[:5]
        o = compare_hands(nh, opp)
        if   o>0: w+=1
        elif o<0: l+=1
        else: t+=1
    return (w+0.5*t)/(w+l+t)

def hand_to_features(hand, mask):
    ranks, suits = hand_ranks_suits(hand)
    rank_oh = np.zeros((5,13), dtype=int)
    suit_oh = np.zeros((5,4), dtype=int)
    for i,(r,s) in enumerate(hand):
        rank_oh[i, r-2]=1; suit_oh[i, SUITS.index(s)]=1
    mask_arr = np.array(mask, dtype=int)
    cnt = Counter(ranks)
    num_pairs = sum(1 for v in cnt.values() if v==2)
    num_trips = sum(1 for v in cnt.values() if v==3)
    is_flush = int(len(set(suits))==1)
    is_str,_ = is_straight(ranks)
    cat,_ = hand_rank(hand)
    f = {
        **{f"r{i}_{RANK_ORDER[j]}": int(rank_oh[i,j]) for i in range(5) for j in range(13)},
        **{f"s{i}_{s}": int(suit_oh[i, si]) for i in range(5) for si,s in enumerate(SUITS)},
        **{f"mask{i}": int(mask_arr[i]) for i in range(5)},
        "num_pairs": num_pairs, "num_trips": num_trips,
        "is_flush": is_flush, "is_straight": int(is_str), "category": cat
    }
    return f

def features_df_from_samples(samples):
    return pd.DataFrame([{**s["features"], "win_prob": s["win_prob"]} for s in samples])

def generar_dataset_montecarlo(n_manos=800, sims_por_opcion=400, seed=42):
    rng = random.Random(seed)
    all_cards = [(r,s) for r in RANK_TO_INT.values() for s in SUITS]
    samples=[]
    for _ in range(n_manos):
        rng.shuffle(all_cards)
        hand = all_cards[:5]
        for mask in discard_masks_allowed(hand):
            wp = simulate_win_probability(hand, mask, sims=sims_por_opcion, rng=rng)
            samples.append({"features": hand_to_features(hand, mask), "win_prob": wp})
    return features_df_from_samples(samples)

def entrenar_modelo_politica(df_mc):
    X = df_mc.drop(columns=["win_prob"]); y = df_mc["win_prob"].values
    model = RandomForestRegressor(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=63)
    model.fit(X,y); return model, list(X.columns)

# -------- RESULT case-insensitive ----------
def _find_label_col(df):
    for c in df.columns:
        if c.lower() == "result":
            return c
    raise ValueError(f"No se encontró la columna 'result' en {list(df.columns)[:25]}...")

# --- Compatibilidad OneHotEncoder (scikit-learn >=1.4 usa sparse_output) ---
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def entrenar_win_model_desde_dataset(train_path, test_path):
    df_tr = pd.read_csv(train_path)
    df_te = pd.read_csv(test_path)

    ycol_tr = _find_label_col(df_tr)
    ycol_te = _find_label_col(df_te)
    print(f"[WIN MODEL] Etiqueta detectada — train:'{ycol_tr}' | test:'{ycol_te}'")

    y_tr = df_tr[ycol_tr].astype(int).values; X_tr = df_tr.drop(columns=[ycol_tr])
    y_te = df_te[ycol_te].astype(int).values; X_te = df_te.drop(columns=[ycol_te])

    num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in X_tr.columns if c not in num_cols]

    pre = ColumnTransformer([
        ("cat", make_ohe(), cat_cols),
        ("num", "passthrough", num_cols)
    ], remainder="drop")

    clf = RandomForestClassifier(n_estimators=400, min_samples_leaf=2, n_jobs=-1, random_state=63)
    pipe = Pipeline([("pre", pre), ("rf", clf)]).fit(X_tr, y_tr)

    # Métricas (si fallan por columnas desalineadas, solo avisamos)
    try:
        p = pipe.predict_proba(X_te)[:,1]
        print(f"[WIN MODEL] Test AUC={roc_auc_score(y_te,p):.3f}  "
              f"ACC={accuracy_score(y_te,(p>=0.5).astype(int)):.3f}  "
              f"LogLoss={log_loss(y_te,np.clip(p,1e-6,1-1e-6)):.3f}")
    except Exception as e:
        print("[WIN MODEL] Aviso métricas:", e)

    joblib.dump(pipe, WIN_MODEL_PATH); print(f"[WIN MODEL] Guardado en: {WIN_MODEL_PATH}")
    return pipe

# -------- Uso del best_model*.pkl del usuario (opcional) ----------
def cargar_best_model():
    if BEST_MODEL_PATH and os.path.exists(BEST_MODEL_PATH):
        try: return joblib.load(BEST_MODEL_PATH)
        except Exception as e: print("[BEST MODEL] No se pudo cargar:", e)
    else:
        print("[BEST MODEL] (opcional) no encontrado.")
    return None

def build_features_for_best_model(best_model, hand):
    """
    Construye un DataFrame con EXACTAMENTE las columnas y el orden que
    el modelo espera (feature_names_in_), para evitar errores de forma.
    """
    if not hasattr(best_model, "feature_names_in_"): return None
    cols = list(best_model.feature_names_in_)
    base = {c:0 for c in cols}
    for i,(r,s) in enumerate(hand):
        rname = INT_TO_RANK[r]
        if f"r{i}_{rname}" in base: base[f"r{i}_{rname}"]=1
        if f"s{i}_{s}" in base:     base[f"s{i}_{s}"]=1
    X = pd.DataFrame([[base[c] for c in cols]], columns=cols)
    return X

def clasificar_jugada_con_best_model(best_model, hand):
    if best_model is None: return None
    try:
        X = build_features_for_best_model(best_model, hand)
        if X is None: return None
        if hasattr(best_model,"predict_proba"):
            probs = best_model.predict_proba(X)[0]
            classes = getattr(best_model,"classes_", np.arange(len(probs)))
            return {"pred": classes[np.argmax(probs)], "probas": dict(zip(classes, probs))}
        return {"pred": best_model.predict(X)[0], "probas": None}
    except Exception as e:
        print("[BEST MODEL] No usable en esta mano:", e); return None

# --------- Desempate pro-kicker alto (y preferir menos descartes) ----------
def _kept_kicker_value(hand, mask):
    ranks = [r for r,_ in hand]
    cnt = Counter(ranks)
    involved = {r for r,c in cnt.items() if c>=2}  # parte de parejas/tríos/full
    kept = [r for i,(r,_) in enumerate(hand) if r not in involved and not mask[i]]
    return max(kept) if kept else -1

# -------- Pipeline completo ----------
def pipeline_entrenamiento(n_manos=800, sims_por_opcion=400, seed=42):
    print("==> 1) Entrenando modelo de prob. de victoria con TU DATASET...")
    win_model = entrenar_win_model_desde_dataset(TRAIN_PATH, TEST_PATH)
    print("==> 2) Generando dataset Montecarlo...")
    df_mc = generar_dataset_montecarlo(n_manos=n_manos, sims_por_opcion=sims_por_opcion, seed=seed)
    df_mc.to_parquet(DATASET_OUT_PATH, index=False)
    print(f"[MC DATASET] -> {DATASET_OUT_PATH}  filas={len(df_mc)}")
    print("==> 3) Entrenando modelo de política...")
    policy_model, feat_names = entrenar_modelo_politica(df_mc)
    joblib.dump({"model": policy_model, "feature_names": feat_names}, POLICY_MODEL_PATH)
    print(f"[POLICY MODEL] Guardado en: {POLICY_MODEL_PATH}")
    return win_model, {"model": policy_model, "feature_names": feat_names}

def cargar_policy_model():
    if os.path.exists(POLICY_MODEL_PATH):
        try:
            b = joblib.load(POLICY_MODEL_PATH); return b["model"], b["feature_names"]
        except Exception as e: print("[POLICY MODEL] No se pudo cargar:", e)
    return None, None

def evaluar_opciones_con_policy(hand, masks, policy_model, feature_names):
    rows = [hand_to_features(hand,m) for m in masks]
    X = pd.DataFrame(rows)
    for c in feature_names:
        if c not in X.columns: X[c]=0
    X = X[feature_names]
    return policy_model.predict(X)

def recomendar_descartes(hand, sims_por_opcion=2000, seed=123):
    masks = discard_masks_allowed(hand)
    policy_model, feats = cargar_policy_model()
    rng = random.Random(seed)
    resultados=[]
    if policy_model is not None:
        preds = evaluar_opciones_con_policy(hand, masks, policy_model, feats)
        for m,p in zip(masks,preds): resultados.append((m,float(p),"ML"))
    else:
        for m in masks:
            p = simulate_win_probability(hand, m, sims=sims_por_opcion, rng=rng)
            resultados.append((m,float(p),"MC"))
    # Orden principal por probabilidad
    resultados.sort(key=lambda x:x[1], reverse=True)

    # --- Desempate pro-kicker alto y preferir menos descartes, si están muy cerca ---
    best_p = resultados[0][1]
    EPS = 0.005  # 0.5 puntos porcentuales
    candidatos = [r for r in resultados if r[1] >= best_p - EPS]
    candidatos.sort(key=lambda x: (_kept_kicker_value(hand, x[0]), -sum(x[0])), reverse=True)
    # Reconstruimos ranking con el mejor desempate al frente
    resto = [r for r in resultados if r not in candidatos]
    return candidatos + resto  # lista (mask, prob, fuente)

# ======== UTILIDADES PARA JUGAR VARIAS RONDAS (interactivo) ========
def prob_ganar_vs_rival(hand, sims=2000):
    """Probabilidad de ganar con la mano actual contra 1 rival (sin descartar)."""
    return simulate_win_probability(hand, (False, False, False, False, False), sims=sims, rng=random.Random())

def _mask_to_str(mask):
    return "".join(["X" if b else "-" for b in mask]) + "  (X=descartar)"

def _cards_to_str(hand, sep=" "):
    return sep.join(format_card(c) for c in hand)

def _parse_hand_input(s):
    """
    Entrada OFICIAL: cartas separadas por ESPACIOS, p.ej. 'AH 3D 5S 3C QH'.
    (Se toleran comas pegadas por accidente al parsear cada carta).
    """
    tokens = [t for t in s.strip().split() if t.strip()]
    if len(tokens) != 5:
        raise ValueError("❌ Debes introducir exactamente 5 cartas separadas por espacio (ej.: AH 3D 5S 3C QH).")
    return [parse_card(t) for t in tokens]

def _parse_draw_input(s, expected):
    tokens = [t for t in s.strip().split() if t.strip()]
    if len(tokens) != expected:
        raise ValueError(f"Debes introducir exactamente {expected} cartas nuevas, separadas por espacio.")
    return tokens

def jugar_rondas(sims_por_opcion=1200):
    """
    Juego vs 1 rival (sin comas):
    1) Pide 5 cartas 'AH 3D 5S 3C QH'
    2) P(ganar) sin descartar
    3) Recomienda descarte y LISTA LAS CARTAS A DESCARTAR
    4) Si hay descartes: el usuario introduce EXACTAMENTE las cartas robadas (espacio)
    5) Muestra mano final, clasifica y da P(ganar) final
    6) Pregunta si se juega otra ronda (S/N)
    """
    rng = random.Random()
    print("Juego vs 1 rival. Formato cartas: AH KD 7C 7D 2S (A,K,Q,J,T=10; H,D,C,S)\n")
    while True:
        # 1) Entrada de mano (espacios)
        hand_str = input("Añade 5 cartas: ").strip()
        try:
            hand = _parse_hand_input(hand_str)
        except Exception as e:
            print(e); print(); continue

        print("Tu mano:", _cards_to_str(hand))

        # 2) P sin descartar
        p_sin = prob_ganar_vs_rival(hand, sims=1200)
        print(f"P(ganar) SIN descartar (MC): {p_sin:.3f}")

        # 3) Recomendación de descarte
        ranking = recomendar_descartes(hand, sims_por_opcion=sims_por_opcion, seed=123)
        best_mask, p_est, fuente = ranking[0]
        mask_str = _mask_to_str(best_mask)

        if all(b is False for b in best_mask):
            print(f"No se recomienda descartar. Mejor opción: {mask_str} | P(ganar) esperada: {p_est:.3f} [{fuente}]")
            final_hand = hand
        else:
            to_discard = [ (i,c) for i,(c,b) in enumerate(zip(hand, best_mask)) if b ]
            print(f"Mejor descarte: {mask_str} | P(ganar) esperada: {p_est:.3f} [{fuente}]")
            print("Las cartas a descartar son:", " ".join(format_card(c) for _,c in to_discard))

            # 4) Robo MANUAL
            need = sum(best_mask)
            draw_str = input(f"Introduce {need} carta(s) robada(s) (separadas por espacio): ").strip()
            try:
                draw_tokens = _parse_draw_input(draw_str, need)
                final_hand, drawn = apply_discard_and_manual_draw(hand, best_mask, draw_tokens)
                print("Cartas robadas:")
                for pos, card in drawn:
                    print(f"  - Pos {pos+1}: {format_card(card)}")
            except Exception as e:
                print("❌ Error de robo:", e); print()
                resp = input("¿Quieres jugar otra ronda? (S/N): ").strip().lower()
                if resp not in {"s","si","sí"}:
                    print("Fin del juego."); break
                else:
                    print(); continue

        # 5) Clasificación & P final
        best_model = cargar_best_model()
        info_best = clasificar_jugada_con_best_model(best_model, final_hand)
        cat_final, _ = hand_rank(final_hand)
        print("Mano final:", _cards_to_str(final_hand))
        if info_best is not None:
            print("[BEST MODEL] Jugada final:", info_best["pred"])
        else:
            print("Jugada final (evaluador interno):", categoria_nombre(cat_final))

        p_final = prob_ganar_vs_rival(final_hand, sims=2000)
        print(f"P(ganar) de la mano final (MC): {p_final:.3f}\n")

        # 6) ¿Otra ronda?
        resp = input("¿Quieres jugar otra ronda? (S/N): ").strip().lower()
        if resp not in {"s","si","sí"}:
            print("Fin del juego."); break
        print()

# ---- Arranque: si faltan artefactos, entrena (usa los CSV detectados) ----
if not (os.path.exists(POLICY_MODEL_PATH) and os.path.exists(WIN_MODEL_PATH)):
    print(">>> Entrenando artefactos porque no se encontraron modelos previos...")
    # 1) Entrena win_model con tu dataset
    _ = entrenar_win_model_desde_dataset(TRAIN_PATH, TEST_PATH)
    # 2) Genera dataset Montecarlo + entrena política
    df_mc = generar_dataset_montecarlo(n_manos=800, sims_por_opcion=400, seed=42)
    df_mc.to_parquet(DATASET_OUT_PATH, index=False)
    policy_model, feat_names = entrenar_modelo_politica(df_mc)
    joblib.dump({"model": policy_model, "feature_names": feat_names}, POLICY_MODEL_PATH)
    print(f"[MC DATASET] -> {DATASET_OUT_PATH}  |  [POLICY MODEL] -> {POLICY_MODEL_PATH}")
else:
    print(">>> Modelos existentes detectados. Saltando entrenamiento inicial.")

# ====== Para jugar en Colab ======
# jugar_rondas(sims_por_opcion=1200)


[IO] train: /content/train_transformed-2.csv
[IO] test : /content/test_transformed-2.csv
[IO] best : /content/best_model-2.pkl
[IO] out  : /content
>>> Modelos existentes detectados. Saltando entrenamiento inicial.


In [5]:
jugar_rondas(sims_por_opcion=1200)

Juego vs 1 rival. Formato cartas: AH KD 7C 7D 2S (A,K,Q,J,T=10; H,D,C,S)

Añade 5 cartas: 2H 3C 5D 2D TS
Tu mano: 2H 3C 5D 2D TS
P(ganar) SIN descartar (MC): 0.504
Mejor descarte: -XX--  (X=descartar) | P(ganar) esperada: 0.739 [ML]
Las cartas a descartar son: 3C 5D
Introduce 2 carta(s) robada(s) (separadas por espacio): 3H 4H
Cartas robadas:
  - Pos 2: 3H
  - Pos 3: 4H
[BEST MODEL] No usable en esta mano: X has 23 features, but StandardScaler is expecting 21 features as input.
Mano final: 2H 3H 4H 2D TS
Jugada final (evaluador interno): Pareja
P(ganar) de la mano final (MC): 0.495

¿Quieres jugar otra ronda? (S/N): S

Añade 5 cartas: 2H 3D 4C 5S 2D
Tu mano: 2H 3D 4C 5S 2D
P(ganar) SIN descartar (MC): 0.515
Mejor descarte: -XXX-  (X=descartar) | P(ganar) esperada: 0.698 [ML]
Las cartas a descartar son: 3D 4C 5S
Introduce 3 carta(s) robada(s) (separadas por espacio): X
❌ Error de robo: Debes introducir exactamente 3 cartas nuevas, separadas por espacio.

¿Quieres jugar otra ronda? (S/N)